# Interactive Visualization Demo

This notebook demonstrates the `viz` module and `astrowidget` — replacing the old
step-by-step matplotlib workflow (manual helper functions, re-rendering on every
parameter change) with **continuous, interactive visualizations** powered by Panel,
HoloViews, and astrowidget.

## What changed

| Old workflow | New workflow |
|---|---|
| Write `lm_cutout()` helper by hand | `ds.radport.explore()` — one line |
| `plot_da()` then re-run cell with new params | Sliders update the plot in real time |
| `dynamic_spectrum_center_pixel()` helper | `DynamicSpectrumExplorer` with click-to-inspect |
| Manual `imshow` with WCS projection | `astrowidget.SkyWidget` — pan/zoom on real sky |
| ipyaladin + FITS round-trip for overlays | `astrowidget` — zarr → GPU directly, no FITS |
| Re-run cells to change time/freq/position | All parameters are live widgets |

## Requirements

```
pip install 'ovro_lwa_portal[visualization]'
pip install astrowidget
```

Or if using pixi (this repo), the default environment already includes everything.

## 1. Load real data from OSN

Credentials are loaded from the project `.env` file via `python-dotenv`.
The dataset is a 10-time x 10-frequency x 4096x4096 OVRO-LWA observation
hosted on OSN and published under a test DOI.

Launch this notebook with: `pixi run jupyter lab`

In [9]:
import warnings
warnings.filterwarnings("ignore")  # suppress astropy IERS download warnings

from dotenv import load_dotenv
load_dotenv()  # loads .env from project root

import ovro_lwa_portal as ovro
import os

# DOI for the 10t x 10f x 4096x4096 test dataset on caltech1.osn.mghpcc.org
DOI = "10.33569/4q7nb-ahq31"

ds = ovro.open_dataset(
    DOI,
    production=False,
    storage_options={
        "key": os.environ["OSN_KEY"],
        "secret": os.environ["OSN_SECRET"],
    },
    # Rechunk spatial dims to 512x512 for interactive use.
    # The on-disk layout is 4096x4096 (~64 MB per chunk), which means every
    # single-pixel read fetches the full chunk from S3. 512x512 (~1 MB)
    # makes point-pixel operations (dynamic spectra, light curves) ~64x faster.
    chunks={"l": 512, "m": 512},
)

print(f"Loaded: {DOI}")
print(f"  Dims:  {dict(ds.dims)}")
print(f"  Vars:  {list(ds.data_vars)}")
print(f"  Time:  {float(ds.time.values[0]):.6f} – {float(ds.time.values[-1]):.6f} MJD")
print(f"  Freq:  {float(ds.frequency.values[0])/1e6:.1f} – {float(ds.frequency.values[-1])/1e6:.1f} MHz")
print(f"  WCS:   {ds.radport.has_wcs}")
print(f"  Chunks: {ds.SKY.data.chunksize}")

Loaded: 10.33569/4q7nb-ahq31
  Dims:  {'time': 10, 'frequency': 10, 'polarization': 1, 'beam_param': 3, 'l': 4096, 'm': 4096}
  Vars:  ['BEAM', 'SKY', 'wcs_header_str']
  Time:  60454.333363 – 60454.334407 MJD
  Freq:  43.2 – 84.6 MHz
  WCS:   True
  Chunks: (1, 1, 1, 512, 512)


/Users/corderocore/Documents/ovro-lwa-portal/src/ovro_lwa_portal/io.py:687: UserWarning: The specified chunks separate the stored chunks along dimension "l" starting at index 512. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_zarr(store, chunks=chunks, **kwargs)
/Users/corderocore/Documents/ovro-lwa-portal/src/ovro_lwa_portal/io.py:687: UserWarning: The specified chunks separate the stored chunks along dimension "m" starting at index 512. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_zarr(store, chunks=chunks, **kwargs)


## 2. Initialize Panel + HoloViews

The explorers use an LRU cache internally — the first view of each (time, freq)
frame takes ~3s (one S3 chunk read), then it's instant on revisit. No bulk
preload needed.

In [ ]:
import panel as pn
import holoviews as hv

pn.extension("bokeh", comms="default")
hv.extension("bokeh")

## 2. Image Explorer — replaces manual `plot_da()` + re-run workflow

**Old way:** Write a `plot_da()` helper, call it with hardcoded `t_idx`/`f_idx`, 
re-run the cell every time you want a different frame.

**New way:** One line. Sliders for time, frequency, polarization, colormap — all 
update the image in real time. No re-running cells.

In [ ]:
# First render fetches one slice from S3 (~3s), then slider changes are cached
ds.radport.explore_image()

## 3. Dynamic Spectrum Explorer — replaces `dynamic_spectrum_center_pixel()` helper

**Old way:** Write a `dynamic_spectrum_center_pixel()` function, compute the spectrum,
call `plot_dynspec()`, manually pick l/m values, re-run to change position.

**New way:** Adjustable l/m sliders. Click on the waterfall to see the 
spectrum at that time and the light curve at that frequency — linked automatically.
All interactions are instant because the data is preloaded into memory on creation.

In [ ]:
# Dynamic spectrum uses the accessor's batched dask.compute (~2 min first time for S3 data)
# Subsequent l/m changes recompute from S3. Linked clicks are fast (single slice).
ds.radport.explore_dynamic_spectrum(l=0.0, m=0.0)

## 4. Cutout Explorer — replaces manual `lm_cutout()` + grid plotting

**Old way:** Write `lm_cutout()` and `_slice_any_order()` helpers, hardcode center
and extent, build a manual grid of subplots, re-run to shift the center.

**New way:** Adjustable center + extent sliders. Click on the cutout to see the 
light curve and spectrum at that pixel — no helper code needed.

In [ ]:
from ovro_lwa_portal.viz.explorers import CutoutExplorer

# Cutout loads one frame on render (~3s), then cached
explorer = CutoutExplorer(ds, l_center=0.0, m_center=0.0, dl=0.2, dm=0.2)
explorer.panel()

## 5. Sky Viewer (astrowidget) — replaces ipyaladin + FITS round-trip

**Old way (ipyaladin):** Reconstruct WCS from zarr, build `astropy.io.fits.HDUList`
from numpy, serialize FITS, call `aladin.add_fits()` — ~20 lines of boilerplate,
FITS round-trip on every slice change, and a broken sphere overlay.

**New way (astrowidget):** `SkyWidget.set_dataset(ds)` — one line. Reads zarr
natively, sends raw float32 bytes to the GPU. No FITS anywhere in the display path.
Pan/zoom on the celestial sphere at 60fps with RA/Dec coordinate readout.

**Controls:**
- **Drag** to rotate the sphere
- **Scroll** to zoom in/out
- **Hover** shows RA/Dec at cursor
- **Toolbar:** Reset (↻), Pan (✥), Box Zoom (⬚)

In [ ]:
from astrowidget import SkyWidget

sky_widget = SkyWidget()
sky_widget.set_dataset(ds)  # zarr → cached slices → WCS → GPU, no FITS
sky_widget.background_survey = "DSS"  # HiPS tiles behind the radio data
sky_widget

In [ ]:
# Change time/frequency slice — cached, instant after first load
sky_widget.time_idx = 5
sky_widget.freq_idx = 5
freq_mhz = float(ds.coords["frequency"].values[sky_widget.freq_idx]) / 1e6
print(f"Showing time=5, freq=5 ({freq_mhz:.1f} MHz)")

# Change display settings (GPU-only, instant)
# sky_widget.colormap = "viridis"
# sky_widget.stretch = "sqrt"
# sky_widget.show_grid = False

# Switch background survey
# sky_widget.background_survey = "Mellinger"
# sky_widget.background_survey = "WISE"
# sky_widget.background_survey = ""  # disable

# Navigate to a known source
# from astropy.coordinates import SkyCoord
# import astropy.units as u
# sky_widget.goto(SkyCoord.from_name("Cas A"), fov=30 * u.deg)

## 6. Full Dashboard — everything combined

All four explorers in a single tabbed dashboard. One line replaces the entire
old notebook workflow.

In [ ]:
# Full dashboard — each tab loads on demand
ds.radport.explore()

## 7. Using components directly (for custom workflows)

You can also use the data bridge functions directly to build custom
HoloViews/Panel layouts — useful when you need more control than the
pre-built explorers provide.

In [ ]:
import holoviews as hv
from ovro_lwa_portal.viz._data import (
    PreloadedCube,
    sky_image_element,
    dynamic_spectrum_element,
    spectrum_element,
    light_curve_element,
)
from ovro_lwa_portal.viz.components import style_sky_image, style_curve

# PreloadedCube uses LRU cache — first access fetches from S3, rest is cached
cube = PreloadedCube(ds)

img = sky_image_element(cube, time_idx=0, freq_idx=0, robust=True)
spec = spectrum_element(cube, l=0.0, m=0.0, time_idx=0)
lc = light_curve_element(cube, l=0.0, m=0.0, freq_idx=0)

layout = (
    style_sky_image(img, cmap="viridis")
    + style_curve(spec)
    + style_curve(lc)
).cols(2)

layout.opts(shared_axes=False)

## 8. Side-by-side: old vs new

Here's a direct comparison of the sky viewer workflows.

In [ ]:
%%time

# === OLD WAY (ipyaladin) ===
# 1. Extract WCS header string from zarr attrs
# 2. Build astropy Header from string
# 3. Extract 2D slice from zarr
# 4. Downsample with coarsen()
# 5. Clip to percentiles
# 6. Build PrimaryHDU with FITS header
# 7. Adjust NAXIS/CDELT/CRPIX for downsampling
# 8. Create HDUList
# 9. aladin.add_fits(hdul, ...)  ← serializes FITS AGAIN
#
# Total: ~40 lines of boilerplate, FITS round-trip on every slice change,
# and a broken sphere overlay that ipyaladin can't render.

# === NEW WAY (astrowidget) ===
# sky_widget = SkyWidget()
# sky_widget.set_dataset(ds)
# sky_widget.background_survey = "DSS"
#
# Total: 3 lines. zarr → numpy → GPU. No FITS anywhere.
# Pan/zoom at 60fps. Time/freq sliders. Click-to-inspect.

print("Old way: ~40 lines + FITS round-trip + broken overlay")
print("New way: 3 lines, no FITS, working overlay with DSS background")
print()
print('sky_widget = SkyWidget()')
print('sky_widget.set_dataset(ds)')
print('sky_widget.background_survey = "DSS"')